# NexTwin AI — Industrial Digital Twin Platform
## Notebook 06: Energy Consumption & Optimization Model

### Objectives
1. **Load Engineered Dataset**: Load `engineered_energy.csv`.
2. **Predict Thermal Loads & Waste**:
   - Predict `heating_load` and `cooling_load` (Multi-output Regression).
   - Predict `energy_waste_pct` and `energy_optimization_score`.
3. **Train Models**: Train XGBoost and LightGBM regressors.
4. **Build Recommendation Engine**: Formulate rules to suggest building attribute modifications that lower heating/cooling load.
5. **Export Best Models**: Export serialized pipeline as `energy_model.pkl`.

In [1]:
import os
import pickle
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 1. Load Data & Prepare Splits
We load the energy efficiency dataset and divide it into features and multi-output/single-output regression targets.

In [2]:
PROCESSED_DIR = os.path.join("..", "datasets", "processed")
df = pd.read_csv(os.path.join(PROCESSED_DIR, "engineered_energy.csv"))

# Define features
feature_cols = [
    'relative_compactness', 'surface_area', 'wall_area', 'roof_area', 
    'overall_height', 'orientation', 'glazing_area', 'glazing_area_distribution'
]

X = df[feature_cols]
# Multi-output targets: heating and cooling loads
y_loads = df[['heating_load', 'cooling_load']]
# Optimization target
y_waste = df['energy_waste_pct']
y_opt = df['energy_optimization_score']

# Split data (80/20)
X_train, X_test, y_train_loads, y_test_loads = train_test_split(X, y_loads, test_size=0.2, random_state=42)
_, _, y_train_waste, y_test_waste = train_test_split(X, y_waste, test_size=0.2, random_state=42)
_, _, y_train_opt, y_test_opt = train_test_split(X, y_opt, test_size=0.2, random_state=42)

print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")

Training features shape: (614, 8)
Testing features shape: (154, 8)


## 2. Define Preprocessor Pipeline
We scale continuous variables to ensure stable convergence.

In [3]:
preprocessor = StandardScaler()
print("Scaler pipeline initialized.")

Scaler pipeline initialized.


## 3. Train Multi-Output Thermal Load Models
We train XGBoost and LightGBM wrapper pipelines to predict both heating and cooling load simultaneously.

In [4]:
models = {
    "XGBoost Multi-Output": MultiOutputRegressor(XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)),
    "LightGBM Multi-Output": MultiOutputRegressor(LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbose=-1))
}

load_results = {}

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    pipeline.fit(X_train, y_train_loads)
    y_pred = pipeline.predict(X_test)
    
    # Metrics for heating load (col 0) and cooling load (col 1)
    rmse_y1 = np.sqrt(mean_squared_error(y_test_loads.iloc[:, 0], y_pred[:, 0]))
    rmse_y2 = np.sqrt(mean_squared_error(y_test_loads.iloc[:, 1], y_pred[:, 1]))
    r2_y1 = r2_score(y_test_loads.iloc[:, 0], y_pred[:, 0])
    r2_y2 = r2_score(y_test_loads.iloc[:, 1], y_pred[:, 1])
    
    load_results[name] = {
        "Heating R2": r2_y1,
        "Cooling R2": r2_y2,
        "Mean R2": (r2_y1 + r2_y2)/2,
        "Pipeline": pipeline
    }
    
    print(f"=== {name} ===")
    print(f"  Heating Load: R2 = {r2_y1:.4f}, RMSE = {rmse_y1:.4f}")
    print(f"  Cooling Load: R2 = {r2_y2:.4f}, RMSE = {rmse_y2:.4f}")

=== XGBoost Multi-Output ===
  Heating Load: R2 = 0.9981, RMSE = 0.4458
  Cooling Load: R2 = 0.9901, RMSE = 0.9564


=== LightGBM Multi-Output ===
  Heating Load: R2 = 0.9979, RMSE = 0.4724
  Cooling Load: R2 = 0.9871, RMSE = 1.0890


C:\Users\anime\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\anime\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


## 4. Train Energy Waste & Optimization Score Regressors
We fit models to predict `energy_waste_pct` and `energy_optimization_score` directly.

In [5]:
waste_models = {
    "XGBoost Waste": XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
    "LightGBM Waste": LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbose=-1)
}

waste_results = {}
for name, model in waste_models.items():
    pipeline_w = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', model)])
    pipeline_w.fit(X_train, y_train_waste)
    y_pred_w = pipeline_w.predict(X_test)
    r2_w = r2_score(y_test_waste, y_pred_w)
    
    pipeline_o = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', model)])
    pipeline_o.fit(X_train, y_train_opt)
    y_pred_o = pipeline_o.predict(X_test)
    r2_o = r2_score(y_test_opt, y_pred_o)
    
    waste_results[name] = {
        "Waste R2": r2_w,
        "Optimization Score R2": r2_o,
        "Pipeline_Waste": pipeline_w,
        "Pipeline_Opt": pipeline_o
    }
    print(f"{name} - Waste R2: {r2_w:.4f}, Optimization R2: {r2_o:.4f}")

XGBoost Waste - Waste R2: 0.9975, Optimization R2: 0.9978


LightGBM Waste - Waste R2: 0.9951, Optimization R2: 0.9956


C:\Users\anime\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\anime\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


## 5. Industrial Optimization Recommendation Engine
We construct a recommendation class that evaluates building characteristics and suggests parameter adjustments (e.g. lowering glazing area or changing compactness/height ratio) to reduce energy loads.

In [6]:
class EnergyOptimizationAdvisor:
    def __init__(self, load_model):
        self.model = load_model
        
    def advise(self, building_features):
        # Predict baseline loads
        input_df = pd.DataFrame([building_features])
        pred_loads = self.model.predict(input_df)[0]
        base_heating, base_cooling = pred_loads[0], pred_loads[1]
        base_total = base_heating + base_cooling
        
        recommendations = []
        
        # Rule 1: High Glazing
        if building_features['glazing_area'] > 0.25:
            # Simulate lowering glazing to 0.1
            sim_features = building_features.copy()
            sim_features['glazing_area'] = 0.1
            sim_df = pd.DataFrame([sim_features])
            sim_loads = self.model.predict(sim_df)[0]
            savings = base_total - (sim_loads[0] + sim_loads[1])
            if savings > 0:
                recommendations.append({
                    "action": "Reduce glazing area window ratio from current value to 10%",
                    "estimated_thermal_load_savings_kw": round(savings, 2),
                    "priority": "High" if savings > 5 else "Medium"
                })
                
        # Rule 2: Large Roof Area / Height
        if building_features['overall_height'] > 5.0 and building_features['roof_area'] > 200.0:
            sim_features = building_features.copy()
            sim_features['overall_height'] = 3.5
            sim_features['roof_area'] = 150.0
            sim_df = pd.DataFrame([sim_features])
            sim_loads = self.model.predict(sim_df)[0]
            savings = base_total - (sim_loads[0] + sim_loads[1])
            if savings > 0:
                recommendations.append({
                    "action": "Redesign layout for a lower height profile (3.5m) and optimized roof span",
                    "estimated_thermal_load_savings_kw": round(savings, 2),
                    "priority": "Medium"
                })
                
        if len(recommendations) == 0:
            recommendations.append({
                "action": "No immediate building layout modification required. Maintain current thermal envelope.",
                "estimated_thermal_load_savings_kw": 0.0,
                "priority": "Low"
            })
            
        return {
            "baseline_heating_load": round(base_heating, 2),
            "baseline_cooling_load": round(base_cooling, 2),
            "baseline_total_load": round(base_total, 2),
            "recommendations": recommendations
        }

# Instantiate advisor
best_load_model_name = max(load_results, key=lambda k: load_results[k]['Mean R2'])
best_load_pipeline = load_results[best_load_model_name]['Pipeline']

advisor = EnergyOptimizationAdvisor(best_load_pipeline)
sample_building = X_test.iloc[0].to_dict()
advice = advisor.advise(sample_building)
print("Sample building optimization analysis:")
import pprint
pprint.pprint(advice)

Sample building optimization analysis:
{'baseline_cooling_load': np.float32(16.53),
 'baseline_heating_load': np.float32(14.76),
 'baseline_total_load': np.float32(31.29),
 'recommendations': [{'action': 'Reduce glazing area window ratio from current '
                                'value to 10%',
                      'estimated_thermal_load_savings_kw': np.float32(4.45),
                      'priority': 'Medium'}]}


## 6. Export Serialized Models
We bundle the best load, waste, and optimization score pipelines into a wrapper dict and export it.

In [7]:
best_waste_model_name = max(waste_results, key=lambda k: waste_results[k]['Waste R2'])
best_waste_pipeline = waste_results[best_waste_model_name]['Pipeline_Waste']
best_opt_pipeline = waste_results[best_waste_model_name]['Pipeline_Opt']

energy_package = {
    'load_model': best_load_pipeline,
    'waste_model': best_waste_pipeline,
    'optimization_model': best_opt_pipeline
}

model_path = "energy_model.pkl"
with open(model_path, 'wb') as f:
    pickle.dump(energy_package, f)

print(f"Serialized energy models package exported to: {os.path.abspath(model_path)}")

Serialized energy models package exported to: D:\PROJECTS\NexTwinAI\notebooks\energy_model.pkl
